In [4]:
import os
import json
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel
import faiss
from glob import glob

# =========================
# CONFIG
# =========================
EMBED_MODEL = "thenlper/gte-large"     # or any open-source embedding model
CHUNK_FOLDER = "chunks_out"            # folder where your chunk files live
FAISS_INDEX_PATH = "/home/smaniyar_umass_edu/BioNLP_Ontology/other/nlp/RAG_data/faiss_index_privacy_qa.bin"   # faiss file
META_PATH = "/home/smaniyar_umass_edu/BioNLP_Ontology/other/nlp/RAG_data/metadata_privacy_qa_500_encoder.json"            # id → metadata mapping
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH = 16
NORMALIZE = True                       # use cosine similarity (recommended)


# =========================
# LOAD MODEL
# =========================
tokenizer = AutoTokenizer.from_pretrained(EMBED_MODEL, use_fast=True)
model = AutoModel.from_pretrained(EMBED_MODEL).to(DEVICE)
model.eval()


BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 1024, padding_idx=0)
    (position_embeddings): Embedding(512, 1024)
    (token_type_embeddings): Embedding(2, 1024)
    (LayerNorm): LayerNorm((1024,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-23): 24 x BertLayer(
        (attention): BertAttention(
          (self): BertSdpaSelfAttention(
            (query): Linear(in_features=1024, out_features=1024, bias=True)
            (key): Linear(in_features=1024, out_features=1024, bias=True)
            (value): Linear(in_features=1024, out_features=1024, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=1024, out_features=1024, bias=True)
            (LayerNorm): LayerNorm((1024,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, 

In [6]:

import os
import json
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel
import faiss
from glob import glob

# =========================
# CONFIG
# =========================
EMBED_MODEL = "thenlper/gte-large"     # or any open-source embedding model
CHUNK_FOLDER = "chunks_out"            # folder where your chunk files live
FAISS_INDEX_PATH = "/home/smaniyar_umass_edu/BioNLP_Ontology/other/nlp/RAG_data/faiss_index_privacy_qa.bin"   # faiss file
META_PATH = "/home/smaniyar_umass_edu/BioNLP_Ontology/other/nlp/RAG_data/metadata_privacy_qa_500_encoder.json"            # id → metadata mapping
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH = 16
NORMALIZE = True                       # use cosine similarity (recommended)


# =========================
# LOAD MODEL
# =========================
tokenizer = AutoTokenizer.from_pretrained(EMBED_MODEL, use_fast=True)
model = AutoModel.from_pretrained(EMBED_MODEL).to(DEVICE)
model.eval()

def mean_pool(last_hidden, attention_mask):
    mask = attention_mask.unsqueeze(-1).float()
    summed = (last_hidden * mask).sum(dim=1)
    count = mask.sum(dim=1).clamp(min=1e-9)
    return summed / count

def encode_texts(text_list):
    out = []
    for i in range(0, len(text_list), BATCH):
        b = text_list[i:i+BATCH]
        inputs = tokenizer(b, padding=True, truncation=True, return_tensors="pt").to(DEVICE)
        with torch.no_grad():
            output = model(**inputs, return_dict=True)
            emb = mean_pool(output.last_hidden_state, inputs["attention_mask"])
        emb = emb.cpu().numpy().astype("float32")
        out.append(emb)

    embs = np.vstack(out)
    if NORMALIZE:
        faiss.normalize_L2(embs)
    return embs


def load_search_components():
    index = faiss.read_index(FAISS_INDEX_PATH)
    with open(META_PATH, "r", encoding="utf-8") as f:
        metadata = json.load(f)
    return index, metadata


def search(query, topk=5):
    index, metadata = load_search_components()

    q_emb = encode_texts([query])
    scores, ids = index.search(q_emb, topk)

    results = []
    for s, i in zip(scores[0], ids[0]):
        if i == -1:
            continue
        meta = metadata[str(int(i))]
        results.append({
            "id": int(i),
            "score": float(s),
            "file": meta["file"]
        })
    return results
q = "Consider \"Fiverr\"'s privacy policy; how does fiverr protect freelancers' personal information?"
print(search(q, topk=5))


[{'id': 31, 'score': 0.9272541403770447, 'file': '/home/smaniyar_umass_edu/BioNLP_Ontology/other/nlp/RAG_data/Chunks_data/privacy_qa_500_encoder/privacy_qa_Fiverr_chunk6.txt'}, {'id': 29, 'score': 0.9244945645332336, 'file': '/home/smaniyar_umass_edu/BioNLP_Ontology/other/nlp/RAG_data/Chunks_data/privacy_qa_500_encoder/privacy_qa_Fiverr_chunk4.txt'}, {'id': 26, 'score': 0.9177379608154297, 'file': '/home/smaniyar_umass_edu/BioNLP_Ontology/other/nlp/RAG_data/Chunks_data/privacy_qa_500_encoder/privacy_qa_Fiverr_chunk11.txt'}, {'id': 30, 'score': 0.9149594306945801, 'file': '/home/smaniyar_umass_edu/BioNLP_Ontology/other/nlp/RAG_data/Chunks_data/privacy_qa_500_encoder/privacy_qa_Fiverr_chunk5.txt'}, {'id': 25, 'score': 0.9143015742301941, 'file': '/home/smaniyar_umass_edu/BioNLP_Ontology/other/nlp/RAG_data/Chunks_data/privacy_qa_500_encoder/privacy_qa_Fiverr_chunk10.txt'}]
